# __Project: Mandarin ASR with wav2vec 2.0 and CTC__

- This project follows the general workflow of the Hugging Face tutorial [Fine-Tuning XLS-R for Multilingual ASR with 🤗 Transformers](https://huggingface.co/blog/fine-tune-xlsr-wav2vec2). 

- The original tutorial has been adapted to the current Hugging Face Transformers API and to Mandarin speech recognition using the AISHELL-3 corpus.

- The goal of the project is to fine-tune a pretrained XLS-R model for Mandarin automatic speech recognition (ASR) using Connectionist Temporal Classification (CTC). Two output representations are investigated:

1. Chinese characters (Hanzi)
2. Tone-marked Pinyin syllables

- This notebook covers the first part of the project, including corpus exploration, transcription parsing, train/validation/test preparation, Hugging Face dataset generation, vocabulary construction, and input preprocessing.

- _AI-assisted development was used for code explanation, debugging, and API migration. Python version 3.13.14, Transformers version 5.12.1_

In [23]:
import torch
import torchcodec
import json
from pathlib import Path
from datasets import Dataset, DatasetDict, Audio
import random
from datasets import load_from_disk, DatasetDict
import re
import csv
from transformers import Wav2Vec2CTCTokenizer
from transformers import Wav2Vec2FeatureExtractor
from transformers import Wav2Vec2Processor
import torch

from dataclasses import dataclass
from typing import Any
from transformers import Wav2Vec2Processor

print(torch.__version__)
print(torchcodec.__version__)

!pip list | grep transformer

2.15.0
0.17.0
transformers            5.12.1


### __📚Start to Build the CTC Vocab__
- including one Hanzi vocab, one Pinyin vocab and one alphabetical phone vocab.

In [2]:
# get the processed dataset root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "processed_data").exists():
    candidate = (
        PROJECT_ROOT
        / "semester-project-chinese-ASR"
    )

    if (candidate / "processed_data").exists():
        PROJECT_ROOT = candidate
    else:
        raise FileNotFoundError(
            "Cannot find processed_data directory."
        )

DATA_ROOT = PROJECT_ROOT / "aishell3"
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)

Project root: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR
Processed data: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processed_data


In [3]:
# already have dataset, laod the dataset, get the miniset
hanzi_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "aishell3_hanzi"
)

pinyin_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "aishell3_pinyin"
)

print("Full Hanzi:", hanzi_dataset)
print("Full Pinyin:", pinyin_dataset)

Loading dataset from disk:   0%|          | 0/36 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/36 [00:00<?, ?it/s]

Full Hanzi: DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
Full Pinyin: DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})


In [4]:
# sanity check: if the dataset aligned with each other:
splits = ["train", "validation", "test"]

for split in splits:
    assert (
        hanzi_dataset[split]["id"]
        == pinyin_dataset[split]["id"]
    )

print("Full datasets are aligned.")

Full datasets are aligned.


#### Now We Start to Build CTC Vocab. 
- Must have placeholder for "blank", "unk", and "|" to separate repeated phones.

- __Hanzi__: each token contains every character

- __Pinyin__: each token contains pinyin+tone together

- __Phone__: in alphabet a to z, also includes tone 1 to 5 (5 as silence).

---
##### __🀄️ Hanzi Vocab__

In [5]:
# getting the unique characters in the Hanzi set
hanzi_characters = sorted({
    character
    for text in hanzi_dataset["train"]["text"]
    for character in text
})

# vocab to id
hanzi_vocab = {
    character: index
    for index, character in enumerate(hanzi_characters)
}

# adding special tokens for unknown and padding
hanzi_vocab["[UNK]"] = len(hanzi_vocab)
hanzi_vocab["[PAD]"] = len(hanzi_vocab)
hanzi_vocab["|"] = len(hanzi_vocab)

print("Hanzi vocabulary size:", len(hanzi_vocab))
print(list(hanzi_vocab.items())[:20])

Hanzi vocabulary size: 4015
[('一', 0), ('丁', 1), ('七', 2), ('万', 3), ('丈', 4), ('三', 5), ('上', 6), ('下', 7), ('不', 8), ('与', 9), ('丐', 10), ('丑', 11), ('专', 12), ('且', 13), ('丕', 14), ('世', 15), ('丘', 16), ('丙', 17), ('业', 18), ('丛', 19)]


---
##### __📦 Pinyin Vocab__

In [6]:
# getting the pinyin pairs from the phone_set in the dataset
def read_train_pinyin_tokens(phone_set_path):
    pinyin_tokens = []
    reading_train_section = False

# read the file, which contains all the pinyin pairs
    with Path(phone_set_path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "# The phones frequencies of the train set.":
                reading_train_section = True
                continue

            if line == "# The phones frequencies of the test set.":
                break

            if not reading_train_section:
                continue

            if not line or line.startswith("#"):
                continue

            pinyin_token, frequency = line.split()

            if not re.fullmatch(r"[a-z]+[1-5]", pinyin_token):
                raise ValueError(
                    f"Invalid Pinyin token: {pinyin_token}"
                )

            pinyin_tokens.append(pinyin_token)

    if len(pinyin_tokens) != len(set(pinyin_tokens)):
        raise ValueError("Duplicate Pinyin tokens found in training section.")

    return sorted(pinyin_tokens)

In [7]:
pinyin_tokens = read_train_pinyin_tokens(DATA_ROOT / "phone_set.txt")
pinyin_vocab = {token: index for index, token in enumerate(pinyin_tokens)}

pinyin_vocab["|"] = len(pinyin_vocab)
pinyin_vocab["[UNK]"] = len(pinyin_vocab)
pinyin_vocab["[PAD]"] = len(pinyin_vocab)

print("Pinyin vocabulary size:", len(pinyin_vocab))
print(list(pinyin_vocab.items())[:20])

Pinyin vocabulary size: 1847
[('a1', 0), ('a2', 1), ('a3', 2), ('a4', 3), ('a5', 4), ('ai1', 5), ('ai2', 6), ('ai3', 7), ('ai4', 8), ('ai5', 9), ('an1', 10), ('an2', 11), ('an3', 12), ('an4', 13), ('ang1', 14), ('ang2', 15), ('ang4', 16), ('ao1', 17), ('ao2', 18), ('ao3', 19)]


---
##### __🔤 Phone Vocab__

In [8]:
# getting the vocab directly from the alphabet and numbers
phone_tokens = list("abcdefghijklmnopqrstuvwxyz12345")

phone_vocab = {
    token: index
    for index, token in enumerate(phone_tokens)
}

phone_vocab["|"] = len(phone_vocab)
phone_vocab["[UNK]"] = len(phone_vocab)
phone_vocab["[PAD]"] = len(phone_vocab)

print("Phone vocabulary size:", len(phone_vocab))
print(phone_vocab)

Phone vocabulary size: 34
{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '1': 26, '2': 27, '3': 28, '4': 29, '5': 30, '|': 31, '[UNK]': 32, '[PAD]': 33}


In [9]:
vocab_names = ["hanzi", "pinyin", "phone"]

vocab_list = [hanzi_vocab, pinyin_vocab, phone_vocab]

VOCAB_DIR = Path("vocab")
VOCAB_DIR.mkdir(parents=True, exist_ok=True)

for vocab_name, vocabulary in zip(vocab_names, vocab_list):
    vocab_path = VOCAB_DIR / f"{vocab_name}_vocab.json"

    with vocab_path.open("w", encoding="utf-8") as vocab_file:
        json.dump(
            vocabulary,
            vocab_file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"Saved {vocab_name} vocabulary: "
        f"{vocab_path} "
        f"({len(vocabulary)} tokens)"
    )

Saved hanzi vocabulary: vocab/hanzi_vocab.json (4015 tokens)
Saved pinyin vocabulary: vocab/pinyin_vocab.json (1847 tokens)
Saved phone vocabulary: vocab/phone_vocab.json (34 tokens)


#### __⚙️ Start to Build the CTC Tokenizer__

__CTC Tokenization Strategies__

- The Hanzi, Pinyin-character, and Pinyin-syllable vocabularies use different tokenization units. Therefore, they cannot all use the default tokenization behavior in the same way.

- _Hanzi Tokenization and Phone Tokenization_

    Each Chinese character is treated as an individual token. For example:

    ```text
    - 你好 
    → ["你", "好"]

    - ni3 hao3
    → ["n", "i", "3", "|", "h", "a", "o", "3"]

- _Pinyin Tokenization_
    - The character-level Pinyin representation has a small output vocabulary, but produces longer label sequences. 
    - The syllable-level representation has a larger vocabulary, but produces shorter and more linguistically meaningful label sequences.

In [10]:
hanzi_vocab_path = VOCAB_DIR / "hanzi_vocab.json"
assert hanzi_vocab_path.exists(), (
    f"Vocabulary not found: {hanzi_vocab_path.resolve()}"
)

phone_vocab_path = VOCAB_DIR / "phone_vocab.json"
assert phone_vocab_path.exists(), (
    f"Vocabulary not found: {phone_vocab_path.resolve()}"
)

hanzi_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(hanzi_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

phone_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(phone_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

print("Hanzi Tokenizer size:", len(hanzi_tokenizer))
print(f"Vocab file:", hanzi_vocab_path.resolve(), "\n")

print("Phone Tokenizer size:", len(phone_tokenizer))
print("Phone Vocab file:", phone_vocab_path.resolve())

Hanzi Tokenizer size: 4017
Vocab file: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/hanzi_vocab.json 

Phone Tokenizer size: 36
Phone Vocab file: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/phone_vocab.json


In [11]:
class PinyinSyllableCTCTokenizer(
    Wav2Vec2CTCTokenizer
):
    def prepare_for_tokenization(
        self,
        text,
        **kwargs,
    ):
        text = "|".join(
            text.lower().split()
        )

        return text, kwargs

pinyin_tokenizer = (
    PinyinSyllableCTCTokenizer(
        vocab_file=str(
            VOCAB_DIR / "pinyin_vocab.json"
        ),
        unk_token="[UNK]",
        pad_token="[PAD]",
        word_delimiter_token="|",
    )
)

In [12]:
# sanity check: if the tokenizer works well. test with an example
hanzi_example = hanzi_dataset["train"][0]["text"]

hanzi_encoding = hanzi_tokenizer(hanzi_example)

print("Text:", hanzi_example)
print("IDs:", hanzi_encoding.input_ids[:20])
print(
    "Tokens:",
    hanzi_tokenizer.convert_ids_to_tokens(
        hanzi_encoding.input_ids[:20]
    ),
)

Text: 但这一点已经足够表明卷福的超高人气
IDs: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]
Tokens: ['但', '这', '一', '点', '已', '经', '足', '够', '表', '明', '卷', '福', '的', '超', '高', '人', '气']


In [13]:
# Sanity check for the Hanzi tokenizer
hanzi_example = hanzi_dataset["train"][0]["text"]

hanzi_encoding = hanzi_tokenizer(
    hanzi_example
)

hanzi_tokens = (
    hanzi_tokenizer.convert_ids_to_tokens(
        hanzi_encoding.input_ids
    )
)

decoded_text = hanzi_tokenizer.decode(
    hanzi_encoding.input_ids,
    group_tokens=False,
)

print("Text:", hanzi_example)
print("IDs:", hanzi_encoding.input_ids)
print("Tokens:", hanzi_tokens)
print("Decoded:", decoded_text)

assert hanzi_tokens == list(hanzi_example)
assert decoded_text == hanzi_example
assert (
    hanzi_tokenizer.unk_token_id
    not in hanzi_encoding.input_ids
)

print("Hanzi tokenizer sanity check passed.")

Text: 但这一点已经足够表明卷福的超高人气
IDs: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]
Tokens: ['但', '这', '一', '点', '已', '经', '足', '够', '表', '明', '卷', '福', '的', '超', '高', '人', '气']
Decoded: 但这一点已经足够表明卷福的超高人气
Hanzi tokenizer sanity check passed.


In [17]:
# Check and print the tokenization results
pinyin_example_1 = "er2 zi5 wanr2 shou3 ji1"
pinyin_example_2 = "er2zi5wanr2shou3ji1"

tokenizer_list = [pinyin_tokenizer, phone_tokenizer]

for tokenizer in tokenizer_list:
    
    encoding_1 = tokenizer(pinyin_example_1)
    encoding_2 = tokenizer(pinyin_example_2)

    tokens_1 = tokenizer.convert_ids_to_tokens(
        encoding_1.input_ids
    )
    tokens_2 = tokenizer.convert_ids_to_tokens(
        encoding_2.input_ids
    )

    decoded_1 = tokenizer.decode(
        encoding_1.input_ids,
        group_tokens=False,
    )
    decoded_2 = tokenizer.decode(
        encoding_2.input_ids,
        group_tokens=False,
    )

    print(
        f"===== {tokenizer.__class__.__name__} ====="
    )
    print("Input 1:", pinyin_example_1)
    print("Token IDs 1:", encoding_1.input_ids)
    print("Tokens 1:", tokens_1)
    print("Decoded 1:", decoded_1)

    print()

    print("Input 2:", pinyin_example_2)
    print("Token IDs 2:", encoding_2.input_ids)
    print("Tokens 2:", tokens_2)
    print("Decoded 2:", decoded_2)

    print()

===== PinyinSyllableCTCTokenizer =====
Input 1: er2 zi5 wanr2 shou3 ji1
Token IDs 1: [371, 1844, 1816, 1844, 1511, 1844, 1326, 1844, 620]
Tokens 1: ['er2', '|', 'zi5', '|', 'wanr2', '|', 'shou3', '|', 'ji1']
Decoded 1: er2 zi5 wanr2 shou3 ji1

Input 2: er2zi5wanr2shou3ji1
Token IDs 2: [371, 1816, 1511, 1326, 620]
Tokens 2: ['er2', 'zi5', 'wanr2', 'shou3', 'ji1']
Decoded 2: er2zi5wanr2shou3ji1

===== Wav2Vec2CTCTokenizer =====
Input 1: er2 zi5 wanr2 shou3 ji1
Token IDs 1: [4, 17, 27, 31, 25, 8, 30, 31, 22, 0, 13, 17, 27, 31, 18, 7, 14, 20, 28, 31, 9, 8, 26]
Tokens 1: ['e', 'r', '2', '|', 'z', 'i', '5', '|', 'w', 'a', 'n', 'r', '2', '|', 's', 'h', 'o', 'u', '3', '|', 'j', 'i', '1']
Decoded 1: er2 zi5 wanr2 shou3 ji1

Input 2: er2zi5wanr2shou3ji1
Token IDs 2: [4, 17, 27, 25, 8, 30, 22, 0, 13, 17, 27, 18, 7, 14, 20, 28, 9, 8, 26]
Tokens 2: ['e', 'r', '2', 'z', 'i', '5', 'w', 'a', 'n', 'r', '2', 's', 'h', 'o', 'u', '3', 'j', 'i', '1']
Decoded 2: er2zi5wanr2shou3ji1



---
After creating the tokenizer and vocanb, we can combine the audio and the tokenized text together

### __🧪 Processor, combined audio and text together__

- Why Data Processor?

    _Raw audio → Feature Extractor → input_values_

    _Text → Tokenizer → labels_

    Wav2Vec2Processor combines these two components into one consistent interface.

In [18]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16_000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

hanzi_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=hanzi_tokenizer,
)

pinyin_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=pinyin_tokenizer,
)

phone_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=phone_tokenizer,
)

In [19]:
def prepare_dataset(example, processor):
    audio = example["audio"]

    processed = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["text"],
    )

    example["input_values"] = (
        processed["input_values"][0]
    )

    example["input_length"] = len(
        example["input_values"]
    )

    example["labels"] = processed["labels"]

    return example

#{
#     "id": "SSB00050001",
#     "audio": {...},
#     "text": "广州女大学生……",
#     "input_values": [...],
#     "input_length": 123456,
#     "labels": [...]
# }

In [20]:
prepared_hanzi_dataset = hanzi_dataset.map(
    prepare_dataset,
    fn_kwargs={
        "processor": hanzi_processor,
    },
    desc="Preparing Hanzi dataset",
)

# pinyin dataset and phone dataset share the same base dataset
# pinyin and phone just different in tokenization function

# prepared_pinyin_dataset = pinyin_dataset.map(
#     prepare_dataset,
#     fn_kwargs={
#         "processor": pinyin_processor,
#     },
#     desc="Preparing Pinyin-syllable dataset",
# )

# prepared_phone_dataset = pinyin_dataset.map(
#     prepare_dataset,
#     fn_kwargs={
#         "processor": phone_processor,
#     },
#     desc="Preparing Pinyin-character dataset",
# )

Preparing Hanzi dataset:   0%|          | 0/56935 [00:00<?, ? examples/s]

Preparing Hanzi dataset:   0%|          | 0/6327 [00:00<?, ? examples/s]

Preparing Hanzi dataset:   0%|          | 0/24773 [00:00<?, ? examples/s]

In [21]:
prepared_hanzi_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent', 'input_values', 'input_length', 'labels'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent', 'input_values', 'input_length', 'labels'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent', 'input_values', 'input_length', 'labels'],
        num_rows: 24773
    })
})

In [22]:
test_example = prepare_dataset(
    hanzi_dataset["train"][0],
    hanzi_processor,
)

print("Input length:", test_example["input_length"])
print("Number of labels:", len(test_example["labels"]))
print("true labels:", test_example["text"])
print("First input values:",
      test_example["input_values"][:10])
print("Labels:", test_example["labels"])

Input length: 78809
Number of labels: 17
true labels: 但这一点已经足够表明卷福的超高人气
First input values: [ 0.04401884  0.02261921  0.00648973  0.03158857  0.03285171 -0.01215215
 -0.04094664 -0.03844285 -0.00987181  0.02702106]
Labels: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]


#### __📊 Data Collator__: Final preparation before fine-tuning

__What does the Data Collator contain?__
```
DataCollatorCTCWithPadding class
├── processor = hanzi_processor  → Hanzi collator
├── processor = pinyin_processor → Pinyin-syllable collator
└── processor = phone_processor  → Pinyin-character collator
```

__Why do we need Data Collator?__

- The data collator combines a list of individual examples into one batch. It dynamically pads the audio inputs and text labels separately, creates an attention mask for the padded audio, replaces label-padding positions with -100, and returns the tensors that the Trainer passes to the model.

__Basic structure of Data Collator?__
```
Dataset
   ↓
DataLoader selects several examples
   ↓
Data Collator creates a padded batch
   ↓
Trainer passes the batch to the model
   ↓
Model computes CTC loss
```

In [44]:
@dataclass
class DataCollatorCTCWithPadding:
    processor : Wav2Vec2Processor
    padding: bool | str = True

    '''
    Here the only args we give from the Trainer to the Data Collator is the feature of the input.

    features shaped like:
        features = [
            {
                "input_values": [...],
                "labels": [...],
            }]

    In the end, data collator will give back the batches in shape: 
    batch = {
        "input_values": torch.Tensor(...),
        "attention_mask": torch.Tensor(...),
        "labels": torch.Tensor(...),
    }

    One Example Input Contents:
    Input length: 78809
    Number of labels: 17
    true labels: 但这一点已经足够表明卷福的超高人气
    First input values: [ 0.04401884  0.02261921  0.00648973  0.03158857  0.03285171 -0.01215215
    -0.04094664 -0.03844285 -0.00987181  0.02702106]
    Labels: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]

'''

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        # step 1: the data collator must get the audio input
        # input_values: the extracted features in single audiop file, e.g mel or mfcc
        input_features = [{"input_values": feature["input_values"]} for feature in features]

        # step 2: prepare the label data, then combine these two features together
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # step 3: padding the audio and the label
        # padding the audio using the feature extractor, audio padding using 0.0
        batch = (self.processor.feature_extractor.pad(input_features, padding = self.padding, return_tensors= "pt"))

        # padding labels using the tokenizer, label padding using -100
        labels_batch = (self.processor.tokenizer.pad(label_features, padding = self.padding, return_tensors = "pt"))

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        batch["labels"] = labels

        return batch

In [45]:
# creating data collator
hanzi_data_collator = (DataCollatorCTCWithPadding(processor = hanzi_processor, padding= True))

# pinyin_data_collator = (DataCollatorCTCWithPadding(processor = pinyin_processor, padding= True))

# phone_data_collator = (DataCollatorCTCWithPadding(processor = phone_processor, padding= True))

In [ ]:
# sanity check I: if the data collator works well. test with an example, show the shape of the batch and the labels
hanzi_features = [
    prepared_hanzi_dataset["train"][0],
    prepared_hanzi_dataset["train"][1],
]

hanzi_batch = hanzi_data_collator(hanzi_features)

print("input_values shape:", hanzi_batch["input_values"].shape)
print("attention_mask shape:", hanzi_batch["attention_mask"].shape)
print("labels shape:", hanzi_batch["labels"].shape)

print(hanzi_batch["labels"])

# pinyin_batch = pinyin_data_collator([
#     prepared_pinyin_dataset["train"][0],
#     prepared_pinyin_dataset["train"][1],
# ])

# phone_batch = phone_data_collator([
#     prepared_phone_dataset["train"][0],
#     prepared_phone_dataset["train"][1],
# ])

input_values shape: torch.Size([2, 78809])
attention_mask shape: torch.Size([2, 78809])
labels shape: torch.Size([2, 17])
tensor([[ 151, 3512,    0, 2161, 1053, 2739, 3418,  775, 3199, 1600,  440, 2560,
         2410, 3413, 3936,   93, 1879],
        [3624,  664,   15, 2359, 3844,  968, 1145, 3741, 3191,  332, -100, -100,
         -100, -100, -100, -100, -100]])


In [48]:
# sanity check II: assert the batch shape is correct, and the labels are correct
assert (hanzi_batch["input_values"].ndim == 2) # 2d shaped output, batch_size x audio_sequence_length
assert (hanzi_batch["attention_mask"].shape== hanzi_batch["input_values"].shape) # padding to the longest
assert hanzi_batch["labels"].ndim == 2 # 2d shaped output, batch_size x label_sequence_length

# check if the batch size is correct, and the labels are correct
assert (hanzi_batch["input_values"].shape[0]== 2)
assert (hanzi_batch["labels"].shape[0]== 2)

print("Hanzi data collator test passed.")

Hanzi data collator test passed.
